# Experiment No. 07 - StyleGAN Concepts and Style Mixing

Run cells from top to bottom. GPU recommended.

In [ ]:
!pip -q install tensorflow tensorflow-datasets matplotlib

## Aim
Demonstrate intermediate latent mapping, style interpolation/mixing, and stochastic variation.

**Note:** This is a compact educational style-based GAN, not the complete NVIDIA StyleGAN architecture.

In [ ]:
import tensorflow as tf, tensorflow_datasets as tfds, matplotlib.pyplot as plt
ds=tfds.load("celeb_a",split="train")
def prep(x):
 im=tf.image.resize(x["image"],(32,32))
 return (tf.cast(im,tf.float32)-127.5)/127.5
ds=ds.map(prep).take(2000).shuffle(2000).batch(32)

In [ ]:
mapping=tf.keras.Sequential([tf.keras.layers.Input((100,)),tf.keras.layers.Dense(128,activation="relu"),tf.keras.layers.Dense(128)])
G=tf.keras.Sequential([tf.keras.layers.Input((128,)),tf.keras.layers.Dense(8*8*128),tf.keras.layers.Reshape((8,8,128)),
 tf.keras.layers.Conv2DTranspose(64,4,2,padding="same",activation="relu"),
 tf.keras.layers.Conv2D(3,3,padding="same",activation="tanh")])
D=tf.keras.Sequential([tf.keras.layers.Input((16,16,3)),tf.keras.layers.Conv2D(64,4,2,padding="same",activation="relu"),tf.keras.layers.Flatten(),tf.keras.layers.Dense(1)])
bce=tf.keras.losses.BinaryCrossentropy(from_logits=True)
go=tf.keras.optimizers.Adam(.0002); do=tf.keras.optimizers.Adam(.0002)
@tf.function
def step(real):
 n=tf.shape(real)[0]; z=tf.random.normal([n,100])
 with tf.GradientTape() as gt,tf.GradientTape() as dt:
  w=mapping(z); fake=G(w,training=True)
  ro=D(tf.image.resize(real,(16,16)),training=True); fo=D(tf.image.resize(fake,(16,16)),training=True)
  gl=bce(tf.ones_like(fo),fo); dl=bce(tf.ones_like(ro),ro)+bce(tf.zeros_like(fo),fo)
 gv=mapping.trainable_variables+G.trainable_variables; dv=D.trainable_variables
 go.apply_gradients(zip(gt.gradient(gl,gv),gv)); do.apply_gradients(zip(dt.gradient(dl,dv),dv))
 return gl,dl
for e in range(3):
 for real in ds: gl,dl=step(real)
 print("Epoch",e+1,"G loss",float(gl),"D loss",float(dl))

In [ ]:
z1=tf.random.normal([1,100]); z2=tf.random.normal([1,100])
w1=mapping(z1); w2=mapping(z2)
styles=[w1,(w1+w2)/2,w2]
plt.figure(figsize=(9,3))
for i,w in enumerate(styles):
 im=G(w)[0]; plt.subplot(1,3,i+1); plt.imshow((im+1)/2); plt.axis("off")
plt.show()
# Random latent perturbations demonstrate stochastic variation
plt.figure(figsize=(9,3)); base=mapping(tf.random.normal([1,100]))
for i in range(3):
 im=G(base+tf.random.normal([1,128])*.15)[0]
 plt.subplot(1,3,i+1); plt.imshow((im+1)/2); plt.axis("off")
plt.show()

## Result
The notebook demonstrates latent mapping and interpolation. It does not implement true per-layer style modulation, coarse-to-fine control, or spatial noise injection found in StyleGAN.